# H SC 790 · Lab 7 · Nowcast the Right Edge, Then Publish Safely

**Week 8.** Due Thursday, March 18, 11:59 PM · 30 points.

**What earns full credit**
- Runs end to end from a fresh runtime
- Nowcast scored on replayed history
- The checker recovers no suppressed cell
- AI-use note included

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/OWNER/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

`covid_cases.csv` is the COVID-19 registry as it stood at 11:59 PM on March 31, 2026. Reports keep arriving after a case's onset, so the most recent weeks are incomplete. Part 1: build the reporting triangle, nowcast the right edge, and test the nowcast on the past. Part 2: build a small-area table and suppress it so that no hidden cell can be recovered.

In [ ]:
import matplotlib.pyplot as plt

cases = pd.read_csv(COURSE_URL + "lab07/covid_cases.csv", dtype={"zip": str},
                    parse_dates=["onset_date", "specimen_date", "report_date", "received_ts"])
pop = pd.read_csv(COURSE_URL + "lab07/population_by_zip_group.csv", dtype={"zip": str})
urllib.request.urlretrieve(COURSE_URL + "lab07/suppression_checker.py", "suppression_checker.py")
from suppression_checker import check

ASOF = pd.Timestamp("2026-03-31 23:59:59")
START = pd.Timestamp("2025-06-01")          # a Sunday; week 0 starts here
print(len(cases), "cases;", cases.onset_date.isna().mean().round(3), "without an onset date")
cases.head(3)

## 1. Build the reporting triangle

Onset week by delay in weeks (from onset to receipt). Plot the epidemic curve by onset date and by report date, and explain the difference.

In [ ]:
def triangle(asof):
    """Onset week (rows) by delay in weeks (columns, capped at 8), using only reports received by `asof`."""
    c = cases[cases.received_ts <= asof].dropna(subset=["onset_date"]).copy()
    c["onset_wk"] = (c.onset_date - START).dt.days // 7
    c["delay_wk"] = ((c.received_ts.dt.normalize() - c.onset_date).dt.days // 7).clip(upper=8)
    now_wk = (pd.Timestamp(asof).normalize() - START).days // 7
    return (c[c.onset_wk <= now_wk].groupby(['onset_wk', 'delay_wk']).size()
             .unstack(fill_value=0))

tri = triangle(ASOF)
tri.tail(8)

In [ ]:
# YOUR CODE HERE

## 2. Estimate the delay distribution

From complete weeks only. Report the share of cases received by each delay. Then ask whether one distribution fits the whole period: compare it across onset months.

In [ ]:
# YOUR CODE HERE

## 3. Nowcast the last four weeks

Use completeness factors (the slide's method, below). Add an interval, by bootstrap or from past error. Plot it, shading the provisional weeks.

In [ ]:
def nowcast(tri):
    # --- multiplicative nowcast (slide) ---
    done = tri.iloc[:-5]                 # complete weeks
    share = done.sum().cumsum() / done.sum().sum()
    seen = tri.sum(axis=1)
    age = np.minimum(seen.index.max() - seen.index, 4)
    return seen, seen / share.reindex(age).values

seen, nc = nowcast(tri)
pd.DataFrame({"seen": seen, "nowcast": nc.round(1)}).tail(6)

## 4. Replay history

For twelve past dates, nowcast with only the data available then, for the last full onset week. Compare with what that week's count eventually became, and with the naive baseline of taking the count as seen. Report error, bias, and interval coverage. Where did it fail?

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

## 5. Build the small-area table

Counts and rates per 100,000 by ZIP code and group for December 2025 through February 2026 (by specimen date), with relative standard errors. Flag unreliable rates (a relative standard error above 30% is a common rule).

In [ ]:
w = cases[(cases.specimen_date >= "2025-12-01") & (cases.specimen_date <= "2026-02-28")]
tab = pd.crosstab(w.zip, w.race_ethnicity)
# YOUR CODE HERE

## 6. Suppress, then attack your own table

Apply primary suppression (hide counts from 1 to 10) and complementary suppression. `publish` adds the margins the public would see; `check` attacks the result. No hidden cell may be recoverable, and none may be pinned to a narrow range. Start with the slide's row rule and see what the checker says.

In [ ]:
# --- suppression for one row of counts (slide) ---
def suppress_row(row, min_n=11):
    hide = (row > 0) & (row < min_n)   # primary
    if hide.sum() == 1:                # complementary
        hide[row[~hide].idxmin()] = True
    return row.astype(object).where(~hide, '*')

def publish(hidden):
    """The table as published: '*' where hidden is True, true counts elsewhere, with true row and column totals."""
    out = tab.astype(object).where(~hidden, "*")
    out["Total"] = tab.sum(axis=1)
    out.loc["Total"] = list(tab.sum(axis=0)) + [int(tab.values.sum())]
    return out.reset_index().astype(str)

primary = (tab > 0) & (tab < 11)
row_rule = tab.apply(suppress_row, axis=1) == "*"
for name, hidden in [("primary only", primary), ("slide's row rule", row_rule)]:
    rep = check(publish(hidden))
    print(f"{name}: hidden {int(hidden.values.sum())}, passed {rep['passed']}, recovered exactly {len(rep['recovered'])}, narrowed {len(rep['narrow_ranges'])}")

In [ ]:
# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. How good was the nowcast?** Error, bias, and coverage, against the naive baseline. Where did it fail?

*Your answer here.*

**2. Onset date or report date?** Which curve would you publish, and what would you write under it?

*Your answer here.*

**3. What did suppression cost?** How many cells and which groups lost their numbers? What did you merge instead?

*Your answer here.*

**4. What would you tell the board?** Three sentences on this week's trend that are true and that you could defend.

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab07/` of your course repository.